# Train the real Fruit Quality model
This notebook trains the MobileNetV2 classifier using the published Fresh and Rotten Fruits dataset. You need the **Original Image.zip** dataset saved in your Google Drive first.

## 1. Connect Google Drive
Upload the downloaded `Original Image.zip` to a folder in your Drive. The original dataset is large, so use Wi-Fi and ensure you have enough Drive and Colab storage.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Set the ZIP path and extract
Change `ZIP_PATH` to the exact location of your Original Image.zip file in Drive.

In [ ]:
from pathlib import Path
import zipfile

ZIP_PATH = Path('/content/drive/MyDrive/Original Image.zip')  # EDIT THIS if your file is in another folder
EXTRACT_DIR = Path('/content/fruit_dataset')
assert ZIP_PATH.exists(), f'ZIP not found: {ZIP_PATH}. Update ZIP_PATH.'
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(EXTRACT_DIR)
print('Extracted to:', EXTRACT_DIR)
for p in list(EXTRACT_DIR.rglob('*'))[:40]:
    if p.is_dir(): print(p)

## 3. Install training dependencies and get the training script

In [ ]:
!pip -q install tensorflow-cpu==2.17.1 numpy<2 pillow scikit-learn
!wget -q https://raw.githubusercontent.com/Vinodvinum/Fruits_quality_detection/main/train.py -O /content/train.py

## 4. Choose the folder containing the class folders
Find the directory that directly contains folders such as `Fresh Apple`, `Rotten Apple`, `Fresh Banana`, and `Rotten Banana`. Set `DATA_DIR` to that directory.

In [ ]:
# EDIT this after checking the directory listing above.
DATA_DIR = '/content/fruit_dataset'  # Example only: change if class folders are nested deeper
from pathlib import Path
print('Class folders:', [p.name for p in Path(DATA_DIR).iterdir() if p.is_dir()])

## 5. Train the classifier
This uses the original images, a stratified-by-folder 80/20 training/validation split, MobileNetV2 transfer learning, and fine-tuning. Training time depends on the Colab runtime.

In [ ]:
!python /content/train.py --data_dir "$DATA_DIR" --epochs 12 --output_dir /content/models

## 6. Save the trained model to Google Drive and download it

In [ ]:
import shutil
from pathlib import Path
MODEL_DIR = Path('/content/models')
DRIVE_MODEL_DIR = Path('/content/drive/MyDrive/fruit_quality_trained_model')
DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(MODEL_DIR/'fruit_quality.keras', DRIVE_MODEL_DIR/'fruit_quality.keras')
shutil.copy2(MODEL_DIR/'class_names.json', DRIVE_MODEL_DIR/'class_names.json')
print('Saved model files to:', DRIVE_MODEL_DIR)
from google.colab import files
files.download(str(MODEL_DIR/'fruit_quality.keras'))
files.download(str(MODEL_DIR/'class_names.json'))

## 7. Add the two generated files to GitHub
In `Vinodvinum/xai-lime-classroom-demo`, create a folder named `models` and upload `fruit_quality.keras` and `class_names.json` inside it. Then Streamlit Community Cloud will redeploy and the image-upload app can load the trained model. If GitHub refuses the model due to its file size, do not force it; use external model hosting instead.